# Survival Analysis

## TCGA-BRCA Molecular Subtypes

This notebook performs Kaplan-Meier survival analysis for transcriptomics-derived breast cancer subtypes identified using unsupervised clustering.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter

In [ ]:
#Load Survival Dataset
df = pd.read_csv(
    "/home/sonia/bioinformatics/cancer-subtyping-multiomics/data/gene_counts/survival_data.csv"
)

df.head()

In [ ]:
#Dataset Overview

print("Dataset shape:", df.shape)

print("\nSubtype counts:\n")
print(df["Subtype"].value_counts())

print("\nEvent counts:\n")
print(df["event"].value_counts())

In [ ]:
#Clean Data
df["time"] = pd.to_numeric(df["time"], errors="coerce")
df["event"] = pd.to_numeric(df["event"], errors="coerce")

df = df.dropna(subset=["time", "event"])

print("Cleaned shape:", df.shape)

In [ ]:
#Kaplan-Meier Survival Curves
kmf = KaplanMeierFitter()

plt.figure(figsize=(8,6))

for subtype in df["Subtype"].unique():

    subset = df[df["Subtype"] == subtype]

    kmf.fit(
        durations=subset["time"],
        event_observed=subset["event"],
        label=subtype
    )

    kmf.plot_survival_function()

plt.title("Kaplan-Meier Survival Analysis")
plt.xlabel("Days")
plt.ylabel("Survival Probability")

plt.show()

In [ ]:
#Survival Summary
df.groupby("Subtype")["time"].describe()

In [ ]:
#Event Distribution by Subtype
df.groupby("Subtype")["event"].sum()

# Interpretation

Kaplan-Meier analysis revealed subtype-associated differences in patient survival outcomes.

Luminal tumors generally demonstrated improved survival trends compared with basal-like tumors, consistent with established breast cancer biology.

These findings support the clinical relevance of transcriptomics-based subtype classification.

# Log-Rank Statistical Test

The log-rank test evaluates whether survival distributions differ significantly between molecular subtypes.

In [ ]:
from lifelines.statistics import logrank_test

luminal = df[df["Subtype"] == "Luminal A"]
basal = df[df["Subtype"] == "Basal-like"]

results = logrank_test(
    luminal["time"],
    basal["time"],
    event_observed_A=luminal["event"],
    event_observed_B=basal["event"]
)

print(results.summary)

# Cox Proportional Hazards Model

The Cox proportional hazards model estimates the association between subtype classification and survival risk.

In [ ]:
cox_df = df.copy()

cox_df["Subtype_numeric"] = cox_df["Subtype"].map({
    "Luminal A": 0,
    "Luminal B": 1,
    "Basal-like": 2
})

cox_df = cox_df.dropna(subset=["Subtype_numeric"])

cox_df.head()

In [ ]:
#Fit Cox Model
from lifelines import CoxPHFitter

cph = CoxPHFitter()

cph.fit(
    cox_df[["time", "event", "Subtype_numeric"]],
    duration_col="time",
    event_col="event"
)

cph.print_summary()

# Hazard Ratios

Hazard ratios estimate the relative survival risk associated with subtype classification.

Higher hazard ratios indicate increased mortality risk.

In [ ]:
#extract hazard ratios
hazard_ratios = pd.DataFrame({
    "Hazard Ratio": cph.hazard_ratios_
})

hazard_ratios

In [ ]:
#Forest Plot of Hazard Ratios
hazard_ratios.plot.bar(figsize=(6,4))

plt.ylabel("Hazard Ratio")
plt.title("Subtype-associated Hazard Ratios")

plt.show()

# Interpretation

The Cox proportional hazards model suggested subtype-associated differences in survival risk.

Basal-like tumors demonstrated increased hazard relative to luminal tumors, consistent with their aggressive biological behavior.

These findings support the prognostic relevance of transcriptomic subtype classification in breast cancer.